In [1]:
# Scenario: Fraud Detection in Customer Support Chats
# - Problem: Fraudsters often contact bank support pretending to be customers, trying to reset passwords or gain access.
# - Challenge: Analysts can’t manually read millions of chat transcripts.
# - Solution: Use LoRA fine‑tuning on a pretrained language model (like distilbert-base-uncased) to classify chats as:
# - 0 → Normal inquiry
# - 1 → Fraud attempt
# - 2 → Suspicious but unclear

In [2]:
pip install transformers datasets peft torch accelerate scikit-learn

In [3]:
import pandas as pd

data = {
    "chat": [
        "Hello I forgot my password can you help me reset it",
        "I am the account owner please change the phone number immediately",
        "What are the bank's working hours",
        "I lost my card and need to verify identity",
        "Reset password now I cannot access my account urgent"
    ],
    "label": [0, 1, 0, 2, 1]  # 0 normal, 1 fraud, 2 suspicious
}

df = pd.DataFrame(data)
print(df)

                                                chat  label
0  Hello I forgot my password can you help me res...      0
1  I am the account owner please change the phone...      1
2                  What are the bank's working hours      0
3         I lost my card and need to verify identity      2
4  Reset password now I cannot access my account ...      1


In [4]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

In [5]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
def tokenize(example):
    return tokenizer(
        example["chat"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize)
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)

In [9]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./fraud_chat_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    save_strategy="epoch"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=3, training_loss=1.0767346223195393, metrics={'train_runtime': 7.7813, 'train_samples_per_second': 1.928, 'train_steps_per_second': 0.386, 'total_flos': 505290493440.0, 'train_loss': 1.0767346223195393, 'epoch': 3.0})

In [11]:
import torch

def detect_chat_fraud(chat):

    inputs = tokenizer(
        chat,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    labels = {
        0: "Normal Inquiry",
        1: "Fraud Attempt",
        2: "Suspicious"
    }

    return labels[prediction]

In [12]:
chat = "I cannot access my account please reset password immediately"

result = detect_chat_fraud(chat)

print("Chat:", chat)
print("Prediction:", result)

Chat: I cannot access my account please reset password immediately
Prediction: Fraud Attempt
